**Solar Panal Defect Detection Using Deeplearning.**

In [ ]:
# Import the required models.

from tensorflow.keras.preprocessing.image import ImageDataGenerator                            # type: ignore
from tensorflow.keras.applications import MobileNetV2 ,preprocess_input                        # type: ignore
from tensorflow.keras.application.ResNet50 import ResNet50                                     # type: ignore
from tensorflow.keras.applications.efficientnet import EfficientNetB0                          # type: ignore
from tensorflow.keras import layers, models                                                    # type: ignore
from sklearn.metrics import classification_report
import tensorflow as tf
import numpy as np


dataset = r"/Users/joe/Downloads/Faulty_solar_panel"                # Dataset path.

# Train generator 
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.3,
    horizontal_flip=True)


# Validation generator 
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                        validation_split=0.2)


train_data = train_datagen.flow_from_directory(dataset,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True)


val_data = val_datagen.flow_from_directory(dataset,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False)


**MobileNetV2 Model Training.**

In [ ]:
# Train the model using MobileNetV2.

base_model = MobileNetV2(input_shape=(224,224,3),include_top=False,weights='imagenet')


base_model.trainable = False               # Freeze the base model.


model_mobilenet = models.Sequential([base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(), 
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(6, activation='softmax')])


model_mobilenet.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])


model_mobilenet.fit(train_data, validation_data=val_data, epochs=18)

In [ ]:
# Classification report for mobilenet.

y_pred_probs = model_mobilenet.predict(val_data)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = val_data.classes

class_names = list(val_data.class_indices.keys())

report = classification_report(y_true, y_pred, target_names=class_names)
print("\nClassification Report:\n")
print(report)

6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 276ms/step

Classification Report:

                   precision    recall  f1-score   support

        Bird-drop       0.82      0.68      0.75        41
            Clean       0.75      0.79      0.77        38
            Dusty       0.64      0.71      0.68        38
Electrical-damage       1.00      0.75      0.86        20
  Physical-Damage       0.61      0.85      0.71        13
     Snow-Covered       0.84      0.88      0.86        24

         accuracy                           0.76       174
        macro avg       0.78      0.78      0.77       174
     weighted avg       0.77      0.76      0.76       174



**ResNet50 Model Training.**

In [ ]:
# Train the model using pretrained ResNet50.

base_model = ResNet50(weights='imagenet',include_top=False,input_shape=(224, 224, 3))


base_model.trainable = False              # Freeze base model


model_res = models.Sequential([base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(6, activation='softmax')  ])


model_res.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])


model_res.fit(train_data, validation_data=val_data, epochs=18)

In [ ]:
# Classification report for resnet50.

y_pred_probs = model_res.predict(val_data)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = val_data.classes

class_names = list(val_data.class_indices.keys())

report = classification_report(y_true, y_pred, target_names=class_names)
print("\nClassification Report:\n")
print(report)

6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 533ms/step

Classification Report:

                   precision    recall  f1-score   support

        Bird-drop       0.44      0.44      0.44        41
            Clean       0.70      0.37      0.48        38
            Dusty       0.44      0.61      0.51        38
Electrical-damage       1.00      0.40      0.57        20
  Physical-Damage       0.83      0.38      0.53        13
     Snow-Covered       0.49      0.96      0.65        24

         accuracy                           0.52       174
        macro avg       0.65      0.53      0.53       174
     weighted avg       0.60      0.52      0.51       174



**EfficientNetB0 Model Training.**

In [ ]:
# Train the model using pretrained EfficientNet.

base_model = EfficientNetB0(weights='imagenet',include_top=False,input_shape=(224,224,3))


base_model.trainable = False


model_eff = models.Sequential([base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(train_data.num_classes, activation='softmax')])


model_eff.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])


model_eff.fit(train_data,validation_data=val_data,epochs=18)




23/23 ━━━━━━━━━━━━━━━━━━━━ 11s 477ms/step - accuracy: 0.9267 - loss: 0.2017 - val_accuracy: 0.8621 - val_loss: 0.4446
Epoch 15/18
23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 493ms/step - accuracy: 0.9351 - loss: 0.1881 - val_accuracy: 0.8621 - val_loss: 0.4430
Epoch 16/18
23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 521ms/step - accuracy: 0.9055 - loss: 0.2647 - val_accuracy: 0.8276 - val_loss: 0.4671
Epoch 17/18
23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 529ms/step - accuracy: 0.9168 - loss: 0.2290 - val_accuracy: 0.8563 - val_loss: 0.4107
Epoch 18/18
23/23 ━━━━━━━━━━━━━━━━━━━━ 12s 534ms/step - accuracy: 0.9450 - loss: 0.1593 - val_accuracy: 0.8506 - val_loss: 0.4398


In [ ]:
# Classification report for efficientnet.

y_pred_probs = model_eff.predict(val_data)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = val_data.classes

class_names = list(val_data.class_indices.keys())

report = classification_report(y_true, y_pred, target_names=class_names)
print("\nClassification Report:\n")
print(report)

6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 362ms/step

Classification Report:

                   precision    recall  f1-score   support

        Bird-drop       0.87      0.80      0.84        41
            Clean       0.73      0.95      0.83        38
            Dusty       0.78      0.76      0.77        38
Electrical-damage       1.00      0.95      0.97        20
  Physical-Damage       1.00      0.62      0.76        13
     Snow-Covered       1.00      0.96      0.98        24

         accuracy                           0.85       174
        macro avg       0.90      0.84      0.86       174
     weighted avg       0.86      0.85      0.85       174



In [ ]:
# Save the trained model.

model_eff.save("EfficientNet_model.keras")

print("Model saved successfully!")

Model saved successfully!
